In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Loading the ENV variables

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Loading the dependencies

In [3]:
# Loading dependencies
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
import feedparser
from urllib.parse import quote
import asyncio
import time

In [4]:
ARXIV_API = "http://export.arxiv.org/api/query"

# Tools

In [5]:
@tool
def fetch_abstract(arxiv_id):
    """
    Fetch the abstract of a research paper from arXiv given its ID.
    """
    url = f"{ARXIV_API}?id_list={arxiv_id.split('/')[-1]}"
    feed = feedparser.parse(url)

    if not feed.entries:
        return None

    entry = feed.entries[0]
    return entry.summary.strip().replace("\n", " ")

# Paper Critique Formats

In [6]:
from pydantic import BaseModel
from typing import Dict

In [7]:
class PaperCritique(BaseModel):
    """Formats for paper critique."""
    summary: str
    strengths: str
    weaknesses: str
    detailed_comments: str

# Reviewer2 Prompts

In [26]:
REVIEWER2_SYSTEM_PROMPT = """
You are Reviewer #2.
Given a paper url, download the abstract, and identify ONE major weakness.
Use fetach_abstract tool to get the abstract from arXiv.
Be concise, critical, but academically professional.
Return your critique in the following format:

summary: <summary of the paper>
strengths: <strengths of the paper>
weaknesses: <weaknesses of the paper>
detailed_comments: <detailed comments on the paper>
"""

# LLM engine

In [28]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# R2 Agent

In [34]:
def make_reviewer2_agent():
    return create_agent(
        model=llm,
        tools=[fetch_abstract],
        system_prompt=REVIEWER2_SYSTEM_PROMPT,
        response_format=PaperCritique
    )

In [35]:
arxiv_paper_url = "https://arxiv.org/pdf/2411.12137"

In [36]:
r2_agent = make_reviewer2_agent()

In [37]:
def capture_paper_reviews(arxiv_paper_url, agent):
    response =  agent.invoke({"messages": arxiv_paper_url, "role":"user"})
    return response["messages"][-1].content

# Capture Critiques

In [38]:
output = capture_paper_reviews(arxiv_paper_url, r2_agent)

In [53]:
from parse_agent_output import format_to_json
import json

In [54]:
key_values = format_to_json(output)
review_dict = json.loads(key_values)

In [55]:
for key in review_dict:
    print(f"{key}: {review_dict[key]}\n")

summary: This paper investigates the impact of data quality issues in deep learning models used for software engineering tasks. It focuses on three types of data: code-based, text-based, and metric-based, and compares models trained on clean datasets versus those with quality issues. The authors analyze the effects of these issues on model performance, identifying symptoms such as biased learning, gradient instability, overfitting, and exploding gradients. The findings are validated using six new datasets, providing insights for practitioners and researchers on improving data monitoring and cleaning methods.

strengths: The paper addresses a significant gap in the literature regarding the impact of data quality on deep learning models in software engineering. It employs a comprehensive empirical investigation and provides valuable insights into the symptoms of data quality issues. The validation of findings using multiple datasets enhances the credibility of the research.

weaknesses: 